In [36]:
import os
# Use 8.3 short paths: the real path contains an accented char (é) which breaks
# Spark's Windows launch scripts when building the Java classpath.
# JDK 17 also fails with IllegalAccessError (module system), so use JRE 8 instead.
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jre1.8.0_503'
os.environ['PATH'] = os.path.join(os.environ['JAVA_HOME'], 'bin') + ';' + os.environ['PATH']
os.environ['SPARK_HOME'] = r'C:\Users\RULISO~1\AppData\Local\Programs\Python\PYTHON~1\Lib\site-packages\pyspark'
os.environ['PYSPARK_PYTHON'] = r'C:\Users\RULISO~1\AppData\Local\Programs\Python\PYTHON~1\python.exe'

import numpy as np
import pandas as pd
import kagglehub
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [37]:
spark = SparkSession.builder \
        .master("local") \
        .appName("atendimento-hospitalar") \
        .config("spark.sql.shuffle.partitions", "2") \
        .config("spark.driver.memory", "1g") \
        .getOrCreate()

print(spark.version)

3.5.9


In [38]:
path = kagglehub.dataset_download('abdulqaderasiirii/hospital-patient-data')
print("Dataset path:", path)

Dataset path: C:\Users\Réulison Silva\.cache\kagglehub\datasets\abdulqaderasiirii\hospital-patient-data\versions\1


In [39]:
excel_path = os.path.join(path, "hospital_data_sampleee.xlsx")

pdf = pd.read_excel(excel_path)
pdf.columns = pdf.columns.str.strip()

# these columns mix float/int/str values, which breaks Spark's row-wise type inference
for col in ["Medication Revenue", "Lab Cost", "Consultation Revenue"]:
    pdf[col] = pd.to_numeric(pdf[col], errors="coerce")

# datetime.time objects aren't a supported Spark type, so convert to strings
for col in ["Entry Time", "Post-Consultation Time", "Completion Time"]:
    pdf[col] = pdf[col].astype(str)

df = spark.createDataFrame(pdf)

df.show(5)

+-------------------+------------------+--------+--------------------+-----------+---------------+------------+----------+----------------------+---------------+----------+
|               Date|Medication Revenue|Lab Cost|Consultation Revenue|Doctor Type|Financial Class|Patient Type|Entry Time|Post-Consultation Time|Completion Time|Patient ID|
+-------------------+------------------+--------+--------------------+-----------+---------------+------------+----------+----------------------+---------------+----------+
|2019-11-04 00:00:00|           1183.22|    10.0|               20.17|     ANCHOR|            HMO|  OUTPATIENT|  08:35:45|              09:17:54|       09:29:46|    C10001|
|2019-11-06 00:00:00|            738.48|     NaN|                15.0|     ANCHOR|      INSURANCE|  OUTPATIENT|  19:19:16|              21:02:36|       21:24:07|    C10002|
|2019-11-02 00:00:00|             660.0|     NaN|               21.17|     ANCHOR|            HMO|  OUTPATIENT|  10:46:52|             

In [40]:
lab_cost = F.col("Lab Cost")

df.where(
    lab_cost.isNotNull() & ~F.isnan(lab_cost)
).show(5)

+-------------------+------------------+--------+--------------------+-----------+---------------+------------+----------+----------------------+---------------+----------+
|               Date|Medication Revenue|Lab Cost|Consultation Revenue|Doctor Type|Financial Class|Patient Type|Entry Time|Post-Consultation Time|Completion Time|Patient ID|
+-------------------+------------------+--------+--------------------+-----------+---------------+------------+----------+----------------------+---------------+----------+
|2019-11-04 00:00:00|           1183.22|    10.0|               20.17|     ANCHOR|            HMO|  OUTPATIENT|  08:35:45|              09:17:54|       09:29:46|    C10001|
|2019-11-13 00:00:00|            570.18|    92.5|                15.0|     ANCHOR|      INSURANCE|  OUTPATIENT|  09:14:45|              10:51:01|       10:51:33|    C10007|
|2019-11-02 00:00:00|            468.02|    10.0|               23.91|      LOCUM|            HMO|  OUTPATIENT|  10:01:25|             

In [41]:
consultation_revenue = F.col("Consultation Revenue")
medication_revenue = F.col("Medication Revenue")

print("Top 50 Consultations by Revenue")

df.where(
    consultation_revenue.isNotNull() & ~F.isnan(consultation_revenue) &
    lab_cost.isNotNull() & ~F.isnan(lab_cost) &
    medication_revenue.isNotNull() & ~F.isnan(medication_revenue)
).orderBy(consultation_revenue.desc()).show(50)


Top 50 Consultations by Revenue
+-------------------+------------------+--------+--------------------+-----------+---------------+------------+----------+----------------------+---------------+----------+
|               Date|Medication Revenue|Lab Cost|Consultation Revenue|Doctor Type|Financial Class|Patient Type|Entry Time|Post-Consultation Time|Completion Time|Patient ID|
+-------------------+------------------+--------+--------------------+-----------+---------------+------------+----------+----------------------+---------------+----------+
|2019-11-03 00:00:00|              15.0|    30.0|               46.34|     ANCHOR|            HMO|  OUTPATIENT|  13:43:27|              14:39:32|       14:41:27|    C24161|
|2019-11-02 00:00:00|              49.5|    10.0|                42.0|     ANCHOR|      CORPORATE|  OUTPATIENT|  16:35:05|              16:46:38|       16:48:14|    C12790|
|2019-11-10 00:00:00|             32.93|     8.0|                40.0|     ANCHOR|      INSURANCE|  OUT

# Sobre os dados

## Os dados são sobre atendimentos de pacientes em um hospital.
## O problema: a clínica recebeu várias reclamações sobre o tempo de espera.
____________
## -- As perguntas que precisamos responder:
### O tipo de paciente afeta o tempo de espera?
### Existe um tipo específico de paciente que espera muito tempo?
### Estamos muito ocupados?
### Temos problemas de dimensionamento de equipe (staffing)?
### Quanto tempo os pacientes esperam antes de serem atendidos pelo médico?
### Que tipo de equipe precisamos, ou onde precisamos dela?
### Quais dias da semana são mais afetados?
### Como podemos resolver isso?

## Preparação dos dados (usando PySpark)
Renomeamos as colunas para `snake_case` e mantemos tudo em um `DataFrame` Spark, já que o objetivo aqui é fazer as consultas com PySpark em vez de pandas.

In [42]:
df_wt = (
    df
    .withColumnRenamed("Date", "date")
    .withColumnRenamed("Medication Revenue", "medication_revenue")
    .withColumnRenamed("Lab Cost", "lab_cost")
    .withColumnRenamed("Consultation Revenue", "consultation_revenue")
    .withColumnRenamed("Doctor Type", "doctor_type")
    .withColumnRenamed("Financial Class", "financial_class")
    .withColumnRenamed("Patient Type", "patient_type")
    .withColumnRenamed("Entry Time", "entry_time")
    .withColumnRenamed("Post-Consultation Time", "post_consultation_time")
    .withColumnRenamed("Completion Time", "completion_time")
    .withColumnRenamed("Patient ID", "patient_id")
)

df_wt.show(2)

+-------------------+------------------+--------+--------------------+-----------+---------------+------------+----------+----------------------+---------------+----------+
|               date|medication_revenue|lab_cost|consultation_revenue|doctor_type|financial_class|patient_type|entry_time|post_consultation_time|completion_time|patient_id|
+-------------------+------------------+--------+--------------------+-----------+---------------+------------+----------+----------------------+---------------+----------+
|2019-11-04 00:00:00|           1183.22|    10.0|               20.17|     ANCHOR|            HMO|  OUTPATIENT|  08:35:45|              09:17:54|       09:29:46|    C10001|
|2019-11-06 00:00:00|            738.48|     NaN|                15.0|     ANCHOR|      INSURANCE|  OUTPATIENT|  19:19:16|              21:02:36|       21:24:07|    C10002|
+-------------------+------------------+--------+--------------------+-----------+---------------+------------+----------+-------------

## EDA / Limpeza

In [43]:
df_wt.printSchema()       # as informações dos dados

root
 |-- date: timestamp (nullable = true)
 |-- medication_revenue: double (nullable = true)
 |-- lab_cost: double (nullable = true)
 |-- consultation_revenue: double (nullable = true)
 |-- doctor_type: string (nullable = true)
 |-- financial_class: string (nullable = true)
 |-- patient_type: string (nullable = true)
 |-- entry_time: string (nullable = true)
 |-- post_consultation_time: string (nullable = true)
 |-- completion_time: string (nullable = true)
 |-- patient_id: string (nullable = true)



In [44]:
n_linhas = df_wt.count()
n_colunas = len(df_wt.columns)
print((n_linhas, n_colunas))

(29998, 11)


In [45]:
duplicadas = n_linhas - df_wt.dropDuplicates().count()
print("Linhas duplicadas:", duplicadas)          # existe algum valor duplicado?

Linhas duplicadas: 0


In [46]:
# existe algum valor nulo?
df_wt.select([
    F.count(F.when(F.col(c).isNull() | (F.isnan(c) if dict(df_wt.dtypes)[c] in ("double", "float") else F.lit(False)), c)).alias(c)
    for c in df_wt.columns
]).show()

+----+------------------+--------+--------------------+-----------+---------------+------------+----------+----------------------+---------------+----------+
|date|medication_revenue|lab_cost|consultation_revenue|doctor_type|financial_class|patient_type|entry_time|post_consultation_time|completion_time|patient_id|
+----+------------------+--------+--------------------+-----------+---------------+------------+----------+----------------------+---------------+----------+
|   0|             11936|   28565|                5576|          0|              0|           0|         0|                     0|              0|         0|
+----+------------------+--------+--------------------+-----------+---------------+------------+----------+----------------------+---------------+----------+



In [47]:
# valores distintos por coluna
df_wt.select([F.approx_count_distinct(c).alias(c) for c in df_wt.columns]).show()

+----+------------------+--------+--------------------+-----------+---------------+------------+----------+----------------------+---------------+----------+
|date|medication_revenue|lab_cost|consultation_revenue|doctor_type|financial_class|patient_type|entry_time|post_consultation_time|completion_time|patient_id|
+----+------------------+--------+--------------------+-----------+---------------+------------+----------+----------------------+---------------+----------+
|  13|              4348|     204|                 250|          3|              5|           1|     20896|                 20696|          21555|     30595|
+----+------------------+--------+--------------------+-----------+---------------+------------+----------+----------------------+---------------+----------+



### Vamos olhar cada coluna

In [48]:
df_wt.groupBy("date").count().orderBy("date").show()

+-------------------+-----+
|               date|count|
+-------------------+-----+
|2019-11-01 00:00:00| 2518|
|2019-11-02 00:00:00| 1471|
|2019-11-03 00:00:00| 1301|
|2019-11-04 00:00:00| 3365|
|2019-11-05 00:00:00| 2798|
|2019-11-06 00:00:00| 2813|
|2019-11-07 00:00:00| 2673|
|2019-11-08 00:00:00| 2405|
|2019-11-09 00:00:00| 1539|
|2019-11-10 00:00:00| 1248|
|2019-11-11 00:00:00| 3617|
|2019-11-12 00:00:00| 2892|
|2019-11-13 00:00:00| 1358|
+-------------------+-----+



In [49]:
df_wt.groupBy("doctor_type").count().orderBy(F.desc("count")).show()

+-----------+-----+
|doctor_type|count|
+-----------+-----+
|     ANCHOR|21913|
|      LOCUM| 6789|
|   FLOATING| 1296|
+-----------+-----+



In [50]:
df_wt.groupBy("financial_class").count().orderBy(F.desc("count")).show()

+---------------+-----+
|financial_class|count|
+---------------+-----+
|      INSURANCE| 9931|
|        PRIVATE| 9121|
|      CORPORATE| 6915|
|            HMO| 3738|
|       MEDICARE|  293|
+---------------+-----+



In [51]:
df_wt.groupBy("patient_type").count().orderBy(F.desc("count")).show()

+------------+-----+
|patient_type|count|
+------------+-----+
|  OUTPATIENT|29998|
+------------+-----+



### (EDA) Não temos o tempo de espera do paciente pronto nos dados, então vamos calculá-lo subtraindo `entry_time` de `completion_time`.
Como as colunas de horário vêm como texto (`HH:mm:ss`), convertemos para segundos com `unix_timestamp` antes de subtrair.

In [52]:
entry_time_sec = F.unix_timestamp("entry_time", "HH:mm:ss")
post_consult_sec = F.unix_timestamp("post_consultation_time", "HH:mm:ss")
completion_sec = F.unix_timestamp("completion_time", "HH:mm:ss")

# soma um dia quando o horário passa da meia-noite, para a diferença não ficar negativa
def _segundos_apos(inicio, fim):
    diferenca = fim - inicio
    return F.when(diferenca < 0, diferenca + 24 * 3600).otherwise(diferenca)

dias_semana_pt = {
    "Sunday": "Domingo", "Monday": "Segunda-feira", "Tuesday": "Terça-feira",
    "Wednesday": "Quarta-feira", "Thursday": "Quinta-feira", "Friday": "Sexta-feira",
    "Saturday": "Sábado",
}
weekday_map = F.create_map([F.lit(x) for pair in dias_semana_pt.items() for x in pair])

df_wt = (
    df_wt
    .withColumn("waiting_ber_munets", F.round(_segundos_apos(entry_time_sec, completion_sec) / 60))
    .withColumn("consultation_period", F.round(_segundos_apos(entry_time_sec, post_consult_sec) / 60, 2))
    .withColumn("process_period", F.round(_segundos_apos(post_consult_sec, completion_sec) / 60, 2))
    .withColumn("weekday", weekday_map[F.date_format("date", "EEEE")])
    .withColumn("hours", F.hour(F.to_timestamp("entry_time", "HH:mm:ss")))
    .withColumn("consultation_perc", F.round(F.col("consultation_period") / F.col("waiting_ber_munets"), 2))
    .withColumn("process_perc", F.round(1 - F.col("consultation_perc"), 2))
)

df_wt.select(
    "patient_id", "waiting_ber_munets", "consultation_period", "process_period",
    "weekday", "hours", "consultation_perc", "process_perc",
).show(5)

+----------+------------------+-------------------+--------------+-------------+-----+-----------------+------------+
|patient_id|waiting_ber_munets|consultation_period|process_period|      weekday|hours|consultation_perc|process_perc|
+----------+------------------+-------------------+--------------+-------------+-----+-----------------+------------+
|    C10001|              54.0|              42.15|         11.87|Segunda-feira|    8|             0.78|        0.22|
|    C10002|             125.0|             103.33|         21.52| Quarta-feira|   19|             0.83|        0.17|
|    C10003|              80.0|              69.55|         10.05|       Sábado|   10|             0.87|        0.13|
|    C10004|              79.0|              77.27|           2.2| Quarta-feira|    9|             0.98|        0.02|
|    C10005|              51.0|              50.47|          0.08|  Sexta-feira|   11|             0.99|        0.01|
+----------+------------------+-------------------+-----

### A primeira pergunta é: o tipo de paciente afeta o tempo de espera? E existe um tipo específico de paciente que espera mais tempo?
`patient_type` só tem um valor único nesse conjunto de dados, então usamos `financial_class` para responder.

In [53]:
def dados_agrupados(coluna):
    """Agrupa por `coluna`, calculando o tempo médio de espera e o total de pacientes."""
    return (
        df_wt.groupBy(coluna)
        .agg(
            F.round(F.avg("waiting_ber_munets")).alias("waiting_ber_munets"),
            F.count("*").alias("number_of_patient"),
        )
        .orderBy(coluna)
        .toPandas()
    )

resposta1 = dados_agrupados("financial_class")
resposta2 = dados_agrupados("weekday")
print(resposta1)
print(resposta2)

  financial_class  waiting_ber_munets  number_of_patient
0       CORPORATE                46.0               6915
1             HMO                46.0               3738
2       INSURANCE                44.0               9931
3        MEDICARE                58.0                293
4         PRIVATE                40.0               9121
         weekday  waiting_ber_munets  number_of_patient
0        Domingo                33.0               2549
1   Quarta-feira                47.0               4171
2   Quinta-feira                42.0               2673
3  Segunda-feira                49.0               6982
4    Sexta-feira                42.0               4923
5         Sábado                43.0               3010
6    Terça-feira                42.0               5690


In [54]:
import plotly.express as px

template_style = "plotly_dark"

fig = px.pie(
    resposta1, values="number_of_patient", names="financial_class", hole=0.6,
    width=600, height=600, template=template_style,
    hover_data=["waiting_ber_munets"],
    labels={"waiting_ber_munets": "tempo de espera em min"},
)
fig.update_traces(textposition="outside", textinfo="percent+label")
fig.show()

## Então: o tipo de paciente afeta o tempo de espera?
#### Observando o gráfico acima por `financial_class`, dá para ver que a proporção de pacientes e o tempo médio de espera variam pouco entre as classes financeiras — nenhuma delas se destaca com um tempo de espera muito maior que as demais, exceto possivelmente `MEDICARE`, que costuma ter volume bem menor e tempos de espera mais altos.
#### Ou seja: o tipo de paciente/classe financeira não parece ter um efeito muito grande no tempo de espera, mas vale monitorar as classes com poucos pacientes, pois elas tendem a ter mais variação.

_________________________________________
## Estamos muito ocupados?
Vamos montar visualizações diárias e por hora para responder a essa pergunta, com dois mapas de calor: um para o tempo de espera e outro para o número de pacientes.

In [55]:
ordem_dias = ["Domingo", "Segunda-feira", "Terça-feira", "Quarta-feira", "Quinta-feira", "Sexta-feira", "Sábado"]

contagem_hora_dia = (
    df_wt.groupBy("hours", "weekday").count()
    .toPandas()
    .pivot(index="hours", columns="weekday", values="count")
    .reindex(columns=ordem_dias)
    .fillna(0)
)

media_espera_hora_dia = (
    df_wt.groupBy("hours", "weekday").agg(F.round(F.avg("waiting_ber_munets"), 1).alias("media"))
    .toPandas()
    .pivot(index="hours", columns="weekday", values="media")
    .reindex(columns=ordem_dias)
    .fillna(0)
)

In [56]:
fig3 = px.imshow(
    contagem_hora_dia,
    labels=dict(x="dia da semana", y="hora", color="número de pacientes"),
    aspect="auto", color_continuous_scale="tempo", template=template_style,
    title="Visualização diária/por hora — número de pacientes",
    text_auto=True, width=700, height=700,
)
fig3.update_xaxes(side="top")
fig3.show()

#### Assumindo que o maior número de pacientes = maior tempo de espera, notamos que existem horários no dia com menos pacientes, como às 7h, 13h, 17h e a partir das 21h.

In [57]:
fig4 = px.imshow(
    media_espera_hora_dia,
    labels=dict(x="dia da semana", y="hora", color="tempo de espera (min)"),
    aspect="auto", color_continuous_scale="tempo", template=template_style,
    title="Visualização diária/por hora — tempo médio de espera",
    text_auto=True, width=700, height=700,
)
fig4.update_xaxes(side="top")
fig4.show()

### Sim, estamos muito ocupados no período da manhã e por volta das 13h
#### Diferente do que o primeiro mapa sugeria, os horários com poucos pacientes (7h, 13h, 17h, 21h+) nem sempre têm tempo de espera baixo — isso indica um possível problema de dimensionamento de equipe nesses horários.

In [58]:
def manha(coluna):
    return (
        df_wt.select("entry_time", "post_consultation_time", "completion_time", "waiting_ber_munets")
        .orderBy(coluna)
        .limit(10)
        .toPandas()
    )

manha("entry_time").head(2)

,entry_time,post_consultation_time,completion_time,waiting_ber_munets
0,07:53:25,08:29:46,08:49:16,56.0
1,07:55:06,08:22:58,08:31:59,37.0


#### Parece que os pacientes entram depois das 7:50, então provavelmente o atendimento (OUTPATIENT) começa às 8:00.

In [59]:
manha("post_consultation_time").head(2)

,entry_time,post_consultation_time,completion_time,waiting_ber_munets
0,07:58:59,08:07:41,08:07:57,9.0
1,08:02:59,08:08:34,08:11:32,9.0


#### Confirma que o atendimento começa por volta das 8:07.

### Vamos montar mais um gráfico para confirmar que é um problema de equipe e não outra coisa.

In [60]:
resposta5 = dados_agrupados("hours")

fig = px.bar(
    x=resposta5["hours"], y=resposta5["waiting_ber_munets"], template=template_style,
    text_auto=".2s", labels={"x": "hora", "y": "tempo de espera (min)"},
).add_traces(
    px.line(
        resposta5, x=resposta5["hours"], text="number_of_patient", y=resposta5["number_of_patient"], markers=True,
    ).update_traces(yaxis="y2", showlegend=True, line=dict(color="red", width=3), name="número de pacientes").data
)
fig.update_layout(yaxis2={"side": "right", "overlaying": "y"})
fig.show()

#### Agora dá para ver que alguns horários têm tempo médio de espera alto mesmo com poucos pacientes — ou seja, sim, é um problema de dimensionamento de equipe.
## Como podemos resolver isso?
#### Aumentando a equipe nesses horários (por exemplo 8h, 9h, 13h, 14h e 18h).

## Que tipo de equipe precisamos, ou onde precisamos dela?
### Temos 4 tipos de marcação de tempo nos dados:
#### `entry_time` = entrada no atendimento (OUTPATIENT)
#### `post_consultation_time` = quando o médico chama o paciente para a sala de consulta
#### `completion_time` = quando o paciente sai da sala de consulta / do prédio
#### `waiting_ber_munets` = tempo total gasto no hospital
__________
Podemos extrair mais informações, como:
#### `consultation_period` = tempo antes de entrar com o médico
#### `process_period` = tempo conversando com o médico
#### `consultation_perc` = % do tempo gasto em `consultation_period`
#### `process_perc` = o restante do percentual de `consultation_period`

# Quanto tempo os pacientes esperam antes de serem atendidos pelo médico?

In [61]:
df_wt.select(F.round(F.avg("consultation_period"), 2).alias("media_consultation_period")).show()

+-------------------------+
|media_consultation_period|
+-------------------------+
|                    38.91|
+-------------------------+



### Os pacientes esperam, em média, esse valor (em minutos) antes de serem atendidos pelo médico.

In [62]:
percentuais = df_wt.select(
    F.round(F.avg("consultation_perc"), 2).alias("consultation_perc"),
    F.round(F.avg("process_perc"), 2).alias("process_perc"),
).toPandas().iloc[0]

periodos = df_wt.select(
    F.round(F.avg("consultation_period")).alias("consultation_period"),
    F.round(F.avg("process_period")).alias("process_period"),
).toPandas().iloc[0]

print(percentuais)

consultation_perc    0.88
process_perc         0.12
Name: 0, dtype: float64


In [63]:
fig = px.pie(
    percentuais, names=percentuais.index, template=template_style, values=percentuais,
    hover_name=periodos.index, labels={"color": "tempo de espera em min"},
)
fig.update_traces(textposition="outside", textinfo="percent+label")
fig.show()

#### Boa parte do tempo do paciente é gasta no `consultation_period` — ou seja, esperando o médico — e apenas uma pequena parte é gasta realmente conversando com o médico.
### O tempo de espera até ver o médico está limitado pelo número de médicos disponíveis.

### A próxima pergunta era: quais dias da semana são mais afetados?

In [64]:
fig2 = px.bar(
    resposta2, x="weekday", y="number_of_patient", color="waiting_ber_munets",
    category_orders={"weekday": ordem_dias},
    labels={"waiting_ber_munets": "tempo de espera (min)"},
    color_continuous_scale=["green", "yellow", "red"],
    template=template_style, title="<b>Visualização diária</b>",
)
fig2.show()

## Para responder: os dias mais afetados costumam ser aqueles com maior volume de pacientes e/ou maior tempo médio de espera (veja o gráfico acima) — normalmente no meio da semana.

# Resumo:
### Pode fazer sentido reforçar a equipe médica nos horários de pico (manhã e início da tarde) e nos dias da semana com maior volume/tempo de espera.

# Ações:
### Avaliar se faz sentido financeiro contratar equipe adicional nesses horários e dias, com base na seção de resumo.

## Estimativas com Machine Learning (Random Forest)
Vamos usar o [`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) do scikit-learn para responder:
- Quantos médicos seriam necessários adicionar para reduzir o volume/tempo de espera em pelo menos 30%?
- Quanto isso aumentaria o `Consultation Revenue`?
- Quanto isso poderia aumentar o número de atendimentos?

**Importante:** os dados não trazem a quantidade real de médicos escalados por horário, apenas o `doctor_type` (categoria, não um identificador de médico). Por isso:
1. Usamos o `RandomForestClassifier` para descobrir **quais fatores mais influenciam** uma hora ter "alta espera" (volume de pacientes, hora do dia, dia da semana, e a carga de trabalho estimada);
2. Como o modelo não pode aprender uma relação causal de "quantos médicos adicionar" sem essa variável nos dados históricos, complementamos com uma **estimativa de capacidade** (baseada na carga de trabalho observada) para responder à pergunta numérica sobre médicos, receita e atendimentos.

Essas são aproximações razoáveis para apoiar a decisão, não uma resposta exata — vamos usar planilha gerada manualmente para simular a escala dos médicos [`dataset/escala_medicos.csv`](../dataset/escala_medicos.csv).

In [65]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# agregamos os dados por (data, hora, dia da semana) para ter uma linha por "janela de 1 hora"
pdf_hora = (
    df_wt.groupBy("date", "hours", "weekday")
    .agg(
        F.count("*").alias("n_pacientes"),
        F.avg("waiting_ber_munets").alias("media_espera"),
        F.sum("process_period").alias("carga_trabalho_min"),
    )
    .toPandas()
)

# carga de trabalho (em médico-horas) necessária para atender a demanda daquela hora sem fila
pdf_hora["carga_trabalho_horas"] = pdf_hora["carga_trabalho_min"] / 60
pdf_hora["medicos_necessarios"] = np.ceil(pdf_hora["carga_trabalho_horas"]).clip(lower=1).astype(int)
pdf_hora["dia_semana_num"] = pdf_hora["weekday"].map({dia: i for i, dia in enumerate(ordem_dias)})

# carrega a escala complementar de médicos
escala_path = os.path.join("dataset", "escala_medicos.csv")
escala_medicos = pd.read_csv(escala_path, parse_dates=["date"]).rename(columns={"hour": "hours"})

# merge com a escala real de médicos
pdf_hora["date"] = pd.to_datetime(pdf_hora["date"])
pdf_hora = pdf_hora.merge(escala_medicos, on=["date", "hours"], how="left")

# rotulamos "alta espera" como acima da mediana geral do tempo de espera
limiar_espera = pdf_hora["media_espera"].median()
pdf_hora["alta_espera"] = (pdf_hora["media_espera"] > limiar_espera).astype(int)

pdf_hora.head()

,date,hours,weekday,n_pacientes,media_espera,carga_trabalho_min,carga_trabalho_horas,medicos_necessarios,dia_semana_num,medicos_escalados,alta_espera
0,2019-11-04,8,Segunda-feira,363,54.537190,15222.56,253.709333,254,1,17,1
1,2019-11-01,11,Sexta-feira,263,44.026616,8291.80,138.196667,139,5,17,1
2,2019-11-05,10,Terça-feira,276,47.257246,1461.31,24.355167,25,2,16,1
3,2019-11-12,16,Terça-feira,107,25.448598,1771.44,29.524000,30,2,11,0
4,2019-11-09,20,Sábado,77,25.376623,291.31,4.855167,5,6,4,0


In [66]:
features = ["n_pacientes", "hours", "dia_semana_num", "medicos_necessarios", "medicos_escalados"]
X = pdf_hora[features]
y = pdf_hora["alta_espera"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

modelo_rf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
modelo_rf.fit(X_train, y_train)

print(classification_report(y_test, modelo_rf.predict(X_test)))

importancias = pd.Series(modelo_rf.feature_importances_, index=features).sort_values(ascending=False)
print("Importância de cada fator para prever 'alta espera':")
print(importancias)

              precision    recall  f1-score   support

           0       0.90      0.73      0.81        26
           1       0.77      0.92      0.84        26

    accuracy                           0.83        52
   macro avg       0.84      0.83      0.83        52
weighted avg       0.84      0.83      0.83        52

Importância de cada fator para prever 'alta espera':
n_pacientes            0.330522
hours                  0.316224
medicos_escalados      0.150429
medicos_necessarios    0.134243
dia_semana_num         0.068583
dtype: float64


### O que o modelo nos diz
O `RandomForestClassifier` confirma que o **volume de pacientes** e a **hora do dia** são os fatores que mais explicam uma janela ter "alta espera" — o que reforça as conclusões dos mapas de calor: os horários de pico precisam de mais equipe.

In [67]:
# deficit = quanto a demanda de uma hora ultrapassa a equipe escalada (dataset/escala_medicos.csv)
pdf_hora["deficit"] = (pdf_hora["medicos_necessarios"] - pdf_hora["medicos_escalados"]).clip(lower=0)
horas_criticas = pdf_hora[pdf_hora["deficit"] > 0]
deficit_medio = horas_criticas["deficit"].mean()

# reduzir o deficit medio em 30% e a nossa estimativa de "reduzir a espera em pelo menos 30%"
medicos_adicionais = int(np.ceil(0.30 * deficit_medio))

print(f"Horas criticas (demanda acima da escala): {len(horas_criticas)} de {len(pdf_hora)}")
print(f"Deficit medio nas horas criticas: {deficit_medio:.1f} medicos")
print(f"Medicos adicionais estimados para reduzir a espera em >= 30%: {medicos_adicionais}")


Horas criticas (demanda acima da escala): 140 de 206
Deficit medio nas horas criticas: 72.7 medicos
Medicos adicionais estimados para reduzir a espera em >= 30%: 22


In [68]:
consultation_revenue = F.col("consultation_revenue")

# tempo médio de consulta (process_period) e receita média por atendimento, ignorando nulos/NaN
media_processo_horas = df_wt.select(F.avg("process_period")).first()[0] / 60
media_receita_atendimento = (
    df_wt.filter(consultation_revenue.isNotNull() & ~F.isnan(consultation_revenue))
    .select(F.avg(consultation_revenue)).first()[0]
)

atendimentos_atuais = df_wt.count()
receita_atual = (
    df_wt.filter(consultation_revenue.isNotNull() & ~F.isnan(consultation_revenue))
    .select(F.sum(consultation_revenue)).first()[0]
)

# capacidade adicional (em médico-horas) somada ao longo de todas as janelas de 1 hora observadas
total_horas_periodo = len(pdf_hora)
capacidade_adicional_horas = medicos_adicionais * total_horas_periodo
atendimentos_extras = capacidade_adicional_horas / media_processo_horas
receita_extra = atendimentos_extras * media_receita_atendimento

print(f"Atendimentos atuais: {atendimentos_atuais}")
print(f"Atendimentos extras estimados: {atendimentos_extras:.0f} (+{atendimentos_extras / atendimentos_atuais * 100:.1f}%)")
print(f"Receita atual (Consultation Revenue): {receita_atual:,.2f}")
print(f"Receita extra estimada: {receita_extra:,.2f} (+{receita_extra / receita_atual * 100:.1f}%)")

Atendimentos atuais: 29998
Atendimentos extras estimados: 11530 (+38.4%)
Receita atual (Consultation Revenue): 457,958.79
Receita extra estimada: 216,214.49 (+47.2%)


## Resumo da estimativa
O `RandomForestClassifier` mostra que **volume de pacientes** e **hora do dia** são os principais fatores por trás da alta espera — a equipe adicional deve ser alocada nos horários de pico identificados nos mapas de calor.

Usando `dataset/escala_medicos.csv` como capacidade real por horário, **140 das 206 janelas de 1 hora** analisadas (~68%) ficam acima da escala, com um déficit médio de **72,7 médicos** nessas horas críticas. Para reduzir esse déficit em pelo menos 30%, estimamos a necessidade de **22 médicos adicionais**.

Essa equipe adicional poderia viabilizar um aumento estimado de **~11.530 atendimentos (+38,4%)** e de **R$ 216.214,49 em `Consultation Revenue` (+47,2%)**.

### Ressalvas
Como `dataset/escala_medicos.csv` é sintético (gerado para fins de estudo, com uma semente fixa para reprodutibilidade), os números acima não substituem uma escala real — mas mostram como a análise fica mais robusta com esse dado complementar.